# 02 — Test query: 2-Maison, 5+ pieces each, avg sale price per year

*"A property that has exactly two homes on it with at least 4 bedrooms in each home"* → sales where EXACTLY two rows have `type_local='Maison'` AND `nombre_pieces_principales >= 5` on each.

Assumption: "home" = Maison only (not Appartement). "4 bedrooms" ≈ 5 pieces principales.

In [ ]:
import duckdb, pandas as pd
from pathlib import Path
PARQUET_GLOB = str(Path.cwd().parent / "data" / "parquet" / "year=*" / "*.parquet")
con = duckdb.connect()
con.execute(f"CREATE VIEW dvf AS SELECT * FROM read_parquet('{PARQUET_GLOB}', hive_partitioning=true)")
print("Row count:", con.execute("SELECT COUNT(*) FROM dvf").fetchone()[0])


## Build the sale-level qualifying set

In [ ]:
con.execute('''
CREATE OR REPLACE TABLE qualifying_sales AS
WITH maisons_per_sale AS (
  SELECT id_mutation,
    COUNT(*) AS n_maisons,
    MIN(nombre_pieces_principales) AS min_pieces,
    MAX(nombre_pieces_principales) AS max_pieces,
    SUM(surface_reelle_bati)       AS total_bati
  FROM dvf WHERE type_local = 'Maison' GROUP BY id_mutation
),
sale_meta AS (
  SELECT id_mutation,
         ANY_VALUE(year) AS year,
         ANY_VALUE(valeur_fonciere) AS valeur,
         ANY_VALUE(nature_mutation) AS nature,
         ANY_VALUE(nom_commune) AS commune,
         ANY_VALUE(code_departement) AS dept
  FROM dvf GROUP BY id_mutation
)
SELECT m.*, s.year, s.valeur, s.nature, s.commune, s.dept
FROM maisons_per_sale m JOIN sale_meta s USING (id_mutation)
WHERE m.n_maisons = 2 AND m.min_pieces >= 5
  AND s.nature = 'Vente' AND s.valeur IS NOT NULL
''')
con.execute('SELECT COUNT(*) FROM qualifying_sales').fetchone()

## Answer: raw (no outlier filtering)

In [ ]:
con.execute('''
SELECT year, COUNT(*) AS n_sales,
       ROUND(AVG(valeur))    AS avg_eur,
       ROUND(MEDIAN(valeur)) AS median_eur,
       ROUND(MIN(valeur))    AS min_eur,
       ROUND(MAX(valeur))    AS max_eur
FROM qualifying_sales GROUP BY year ORDER BY year
''').df()

## Why the means are noisy: €1 sales and €200M+ outliers
There are ~80/year sales at valeur=€1 (nominal/symbolic transfers, gifts, divisions) and a handful of €10M+ outliers. Filtering to plausible residential range:

In [ ]:
con.execute('''
SELECT year, COUNT(*) AS n_sales,
       ROUND(AVG(valeur))    AS avg_eur,
       ROUND(MEDIAN(valeur)) AS median_eur
FROM qualifying_sales
WHERE valeur BETWEEN 20000 AND 20000000
GROUP BY year ORDER BY year
''').df()

## Most expensive qualifying sales (sniff test)

In [ ]:
con.execute('''
SELECT year, commune, dept, ROUND(valeur) AS valeur,
       min_pieces, max_pieces, total_bati
FROM qualifying_sales ORDER BY valeur DESC LIMIT 10
''').df()